# Univariate

In [1]:
using LinearAlgebra
using DynamicPolynomials
using MultivariateMoments
using JuMP
using MacaulayMatrix

Consider the following example:

In [2]:
@polyvar x
p = 3x^4 + 8x^3 - 6x^2 + 24x + 1
q = differentiate(p, x)

24 - 12x + 24x² + 12x³

Let's solve that system:

In [3]:
sols = solve_system([q], column_maxdegree = 4)
nothing # hide

[ Info: Added 1 rows to complete columns up to degree 3
[ Info: Nullspace of dimensions (4, 3) computed from Macaulay matrix of dimension (1, 4) in 6.3095e-5 seconds.
[ Info: Found 1 real solution


The real roots are:

In [4]:
sols

1-element Vector{Vector{Float64}}:
 [-2.658967081916994]

With PSD Hankel matrix trick no complex root!

In [5]:
import SCS
solver = SCS.Optimizer
psd_hankel([q], solver, 4)

[ Info: Nullspace of dimensions (5, 3) computed from Macaulay matrix of dimension (2, 5) in 6.7932e-5 seconds.
------------------------------------------------------------------
	       SCS v3.3.1 - Splitting Conic Solver
	(c) Brendan O'Donoghue, Stanford University, 2012
------------------------------------------------------------------
problem:  variables n: 3, constraints m: 7
cones: 	  z: primal zero / dual free vars: 1
	  s: psd vars: 6, ssize: 1
settings: eps_abs: 1.0e-04, eps_rel: 1.0e-04, eps_infeas: 1.0e-07
	  alpha: 1.50, scale: 1.00e-01, adaptive_scale: 1
	  max_iters: 100000, normalize: 1, rho_x: 1.00e-06
	  acceleration_lookback: 10, acceleration_interval: 5
	  compiled with openmp parallelization enabled
lin-sys:  sparse-direct-amd-qdldl
	  nnz(A): 21, nnz(P): 0
------------------------------------------------------------------
 iter | pri res | dua res |   gap   |   obj   |  scale  | time (s)
------------------------------------------------------------------
     0| 1.31

1-element Vector{Vector{Float64}}:
 [-2.658967078140547]

What happened there ?
First, we computed the MacaulayMatrix matrix nullspace

In [6]:
Z = nullspace(macaulay([q], 4))

[ Info: Nullspace of dimensions (5, 3) computed from Macaulay matrix of dimension (2, 5) in 5.6916e-5 seconds.


MultivariateMoments.MacaulayNullspace{Float64, LinearAlgebra.Adjoint{Float64, Matrix{Float64}}, MultivariateBases.MonomialBasis{DynamicPolynomials.Monomial{DynamicPolynomials.Commutative{DynamicPolynomials.CreationOrder}, MultivariatePolynomials.Graded{MultivariatePolynomials.LexOrder}}, DynamicPolynomials.MonomialVector{DynamicPolynomials.Commutative{DynamicPolynomials.CreationOrder}, MultivariatePolynomials.Graded{MultivariatePolynomials.LexOrder}}}}([-0.5648877062932672 -0.5061871608167462 -0.08949732306804505; 0.353091694470675 -0.5869837085383489 -0.30541182230947456; … ; -0.005487708235674736 0.6097339283112615 -0.15500888702836046; 0.048969435176441645 -0.13767209715388368 0.9351372741031582], MonomialBasis([1, x, x², x³, x⁴]), 5.438959822042073e-16)

The PSD hankel matrix we find is:

In [7]:
M = moment_matrix(Z, solver, 2)

------------------------------------------------------------------
	       SCS v3.3.1 - Splitting Conic Solver
	(c) Brendan O'Donoghue, Stanford University, 2012
------------------------------------------------------------------
problem:  variables n: 3, constraints m: 7
cones: 	  z: primal zero / dual free vars: 1
	  s: psd vars: 6, ssize: 1
settings: eps_abs: 1.0e-04, eps_rel: 1.0e-04, eps_infeas: 1.0e-07
	  alpha: 1.50, scale: 1.00e-01, adaptive_scale: 1
	  max_iters: 100000, normalize: 1, rho_x: 1.00e-06
	  acceleration_lookback: 10, acceleration_interval: 5
	  compiled with openmp parallelization enabled
lin-sys:  sparse-direct-amd-qdldl
	  nnz(A): 21, nnz(P): 0
------------------------------------------------------------------
 iter | pri res | dua res |   gap   |   obj   |  scale  | time (s)
------------------------------------------------------------------
     0| 1.31e+00  4.13e-01  1.10e+00  5.50e-01  1.00e-01  1.64e-04 
   175| 3.35e-07  1.81e-11  4.56e-11 -2.28e-11  2.75e-0

MomentMatrix with row/column basis:
 MonomialBasis([1, x, x^2])
And entries in a 3×3 MultivariateMoments.SymMatrix{Float64}:
  1.0000000000564189   -2.6589670479716325    7.070106035693389
 -2.6589670479716325    7.070106035693389   -18.79917911947125
  7.070106035693389   -18.79917911947125     49.98639837057915

From which we get:

In [8]:
atomic_measure(M, 1e-6)

Atomic measure on the variables x with 1 atoms:
 at [-2.658967078140547] with weight 1.000000012344379

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*